# German Replication Diagnostic

This is a separate diagnostic notebook. It does not modify the main replication notebook or any project data files. It checks the executed result files and reports whether they match the stated project checkpoints.


## 1. Locate the project folder

The code searches the current working directory, nearby parent folders, and common child folders for the executed result files.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

EXPECTED_FILES = [
    'germany_crash_results.xlsx',
    'germany_crash_test.xlsx',
    'germany_footnote15_regressions.xlsx',
    'germany_event_time_paths.xlsx',
    'germany_table1_paper_style.xlsx',
    'germany_table3_main.xlsx',
    'germany_replication_checks.xlsx',
    'germany_event_time_paths.png',
]

def find_project_dir():
    cwd = Path.cwd()
    candidates = []

    for base in [cwd, *cwd.parents]:
        candidates.append(base)
        try:
            candidates.extend(item for item in base.iterdir() if item.is_dir())
        except OSError:
            pass

    for base in [cwd, *cwd.parents]:
        for folder_name in ['data', 'project', 'results', 'files', 'input', 'inputs']:
            candidate = base / folder_name
            if candidate.is_dir():
                candidates.append(candidate)

    scored = []
    for candidate in dict.fromkeys(candidates):
        if not candidate.is_dir():
            continue
        present = sum((candidate / filename).exists() for filename in EXPECTED_FILES)
        if present > 0:
            scored.append((present, candidate))

    if not scored:
        raise FileNotFoundError(
            'Could not find the project folder. Run this notebook near the executed Excel files.'
        )

    scored.sort(key=lambda item: (-item[0], str(item[1])))
    return scored[0][1]

PROJECT_DIR = find_project_dir()
display(Markdown(f'**Project folder:** `{PROJECT_DIR}`'))


## 2. Check required files


In [ ]:
missing_files = []

for filename in EXPECTED_FILES:
    path = PROJECT_DIR / filename
    if path.exists():
        print(f'FOUND:   {filename}')
    else:
        print(f'MISSING: {filename}')
        missing_files.append(filename)


## 3. Check run-up episode counts

These are comparisons against the stated project checkpoints. A failed check identifies a difference; it does not change the underlying file.


In [ ]:
def load_excel(filename):
    path = PROJECT_DIR / filename
    if not path.exists():
        return None
    return pd.read_excel(path)

def report(label, passed, details=''):
    status = 'PASS' if passed else 'FAIL'
    suffix = f' - {details}' if details else ''
    print(f'{status}: {label}{suffix}')

crash_results = load_excel('germany_crash_results.xlsx')
expected_runups = {50: 11, 75: 8, 100: 5, 125: 5, 150: 3}

if crash_results is None:
    print('SKIP: germany_crash_results.xlsx is missing')
else:
    threshold_col = 'Pick-up threshold (%)'
    count_col = 'Number of run-ups identified'
    if threshold_col not in crash_results.columns or count_col not in crash_results.columns:
        print('FAIL: crash-results columns are not in the expected format')
    else:
        for threshold, expected in expected_runups.items():
            rows = crash_results[crash_results[threshold_col] == threshold]
            if rows.empty:
                report(f'{threshold}% run-up episodes', False, 'threshold row not found')
            else:
                actual = int(rows.iloc[0][count_col])
                report(
                    f'{threshold}% run-up episodes',
                    actual == expected,
                    f'found {actual}; expected {expected}'
                )


## 4. Check the formal crash test


In [ ]:
crash_test = load_excel('germany_crash_test.xlsx')
expected_crash_test = {
    'Unconditional': (4968, 585),
    '100% run-up': (5, 3),
    'Non-100% months': (4963, 582),
}

if crash_test is None:
    print('SKIP: germany_crash_test.xlsx is missing')
else:
    required_columns = {'Comparison', 'Observations', 'Crashes'}
    if not required_columns.issubset(crash_test.columns):
        print('FAIL: crash-test columns are not in the expected format')
    else:
        for label, (expected_n, expected_crashes) in expected_crash_test.items():
            rows = crash_test[crash_test['Comparison'].astype(str).str.strip() == label]
            if rows.empty:
                report(label, False, 'row not found')
            else:
                row = rows.iloc[0]
                actual_n = int(row['Observations'])
                actual_crashes = int(row['Crashes'])
                report(
                    label,
                    actual_n == expected_n and actual_crashes == expected_crashes,
                    f'found N={actual_n}, crashes={actual_crashes}; expected N={expected_n}, crashes={expected_crashes}'
                )


## 5. Check event-time and regression outputs


In [ ]:
footnote15 = load_excel('germany_footnote15_regressions.xlsx')
if footnote15 is None:
    print('SKIP: germany_footnote15_regressions.xlsx is missing')
else:
    report('Footnote-15 regression rows', len(footnote15) == 20, f'found {len(footnote15)}; expected 20')

event_time = load_excel('germany_event_time_paths.xlsx')
if event_time is None:
    print('SKIP: germany_event_time_paths.xlsx is missing')
else:
    report('Event-time observations', len(event_time) == 73, f'found {len(event_time)}; expected 73')
    if 'event_month' in event_time.columns:
        month_zero = int((event_time['event_month'] == 0).sum())
        report('Event month 0 observations', month_zero == 1, f'found {month_zero}; expected 1')
    else:
        print('FAIL: event_month column is missing')


## 6. Display the executed replication checks


In [ ]:
replication_checks = load_excel('germany_replication_checks.xlsx')
if replication_checks is None:
    print('SKIP: germany_replication_checks.xlsx is missing')
else:
    display(replication_checks)


## 7. Summary

A FAIL result means that the current executed output differs from the stated checkpoint. This diagnostic notebook does not edit, regenerate, or overwrite any project file.


In [ ]:
print('Diagnostic complete.')
if missing_files:
    print('Missing required files:')
    for filename in missing_files:
        print(f'  - {filename}')
else:
    print('All required diagnostic files are present.')
print('The original germany_gsy_replication.ipynb was not modified.')
